In [1]:
import os
import chromadb
from dotenv import load_dotenv
load_dotenv()

VECTORDB_PATH = os.getenv('VECTORDB_PATH', './vectordb')
if not os.path.isabs(VECTORDB_PATH):
    VECTORDB_PATH = os.path.abspath(VECTORDB_PATH)

client = chromadb.PersistentClient(path=VECTORDB_PATH)

# 컬렉션 리스트 조회
def list_collections():
    collections = [c.name for c in client.list_collections()]
    print(f"[INFO] 현재 벡터DB에 존재하는 컬렉션: {collections}")
    return collections

# 각 컬렉션에서 데이터 일부 샘플 조회
def show_sample_from_collection(collection_name, n=3):
    if collection_name not in [c.name for c in client.list_collections()]:
        print(f"[WARN] 컬렉션 '{collection_name}'이 존재하지 않습니다.")
        return
    collection = client.get_collection(collection_name)
    results = collection.get()
    print(f"[INFO] '{collection_name}' 내 데이터 개수: {len(results['ids'])}")
    for i, (doc, meta) in enumerate(zip(results['documents'], results['metadatas'])):
        if i >= n:
            break
        print(f"[{i+1}] doc: {doc}\n    meta: {meta}")


In [2]:
collections = list_collections()
# 각 컬렉션에서 샘플 데이터 3개씩 출력
for cname in collections:
    show_sample_from_collection(cname, n=3) 

[INFO] 현재 벡터DB에 존재하는 컬렉션: ['spo2', 'blood_glucose', 'ldl_cholesterol', 'hdl_cholesterol', 'blood_pressure']
[INFO] 'spo2' 내 데이터 개수: 639
[1] doc: {'total_count': 639, 'id': 6444, 'memb_id': '52a62c11-7c4f-4912-91c9-ff5c145328cd', 'device_desc': '갤럭시워치(Galaxy Watch)', 'fullname': '김테스터', 'get_time': '2025-03-17 17:37:00', 'spo2': 88, 'heart_rate': 0, 'create_dttm': '2025-03-17 15:53:56'}
    meta: {'created_by_system': 'FaMiliCare', 'anonymized': False, 'ingest_time': '2025-12-17 20:38:40', 'is_private': True, 'data_source': '갤럭시워치(Galaxy Watch)', 'consent_granted': True, 'data_type': 'spo2', 'get_time': '2025-03-17 17:37:00', 'memb_id': '52a62c11-7c4f-4912-91c9-ff5c145328cd', 'create_dttm': '2025-03-17 15:53:56', 'device_desc': '갤럭시워치(Galaxy Watch)', 'heart_rate': 0, 'fullname': '김테스터'}
[2] doc: {'total_count': 639, 'id': 6443, 'memb_id': '52a62c11-7c4f-4912-91c9-ff5c145328cd', 'device_desc': '갤럭시워치(Galaxy Watch)', 'fullname': '김테스터', 'get_time': '2025-03-17 05:38:00', 'spo2': 90, 'hear

In [5]:
collection = client.get_collection("blood_pressure")

In [7]:
count = collection.count()
print("blood_pressure 컬렉션의 전체 데이터 건수:", count)

blood_pressure 컬렉션의 전체 데이터 건수: 1186


In [8]:
# 일부 데이터(예: 3개) 조회 - 임베딩과 메타데이터 포함
docs = collection.get(
    include=["embeddings", "metadatas", "documents"],  # 포함할 항목
    limit=1
)
docs

{'ids': ['11104'],
 'embeddings': array([[ 1.65543973e-01, -1.07824743e-01,  5.65329432e-01,
         -1.11731565e+00, -2.24227786e-01, -1.07116461e-01,
          9.46126461e-01, -1.87166944e-01,  4.26917851e-01,
          4.41701300e-02, -1.53284907e-01,  4.39969480e-01,
         -8.52212459e-02,  5.85384607e-01, -2.34096888e-02,
         -2.07942337e-01, -6.16516881e-02,  8.02335143e-02,
          6.90142334e-01,  1.07142888e-01, -4.01777923e-01,
         -2.11815670e-01,  6.75934017e-01,  4.63536978e-01,
         -2.42594525e-01,  5.25452137e-01,  2.10230142e-01,
         -1.63479280e-02,  8.00966248e-02,  6.81275249e-01,
         -6.99507654e-01, -1.86950594e-01, -6.44270301e-01,
         -2.67592166e-02,  2.15746999e-01,  2.10746527e-01,
         -6.83188558e-01, -2.68069208e-01, -3.08457583e-01,
         -4.51953948e-01, -2.78553255e-02,  9.97117639e-01,
          1.24203565e-03,  8.56870294e-01, -1.24667764e-01,
         -2.77231932e-01,  5.01345813e-01, -3.63232911e-01,
       

In [18]:
# 일부 데이터(예: 3개) 조회 - 임베딩과 메타데이터 포함
docs = collection.get(
    include=["embeddings", "metadatas", "documents"],  # 포함할 항목
    limit=3
)

# 결과 출력
for i in range(len(docs['ids'])):
    print(f"[{i}] ID: {docs['ids'][i]}")
    print(f"문서 내용: {docs['documents'][i]}")
    print(f"메타데이터: {docs['metadatas'][i]}")
    print(f"임베딩 벡터 (길이 {len(docs['embeddings'][i])}): {docs['embeddings'][i][:5]}...")  # 일부만 출력
    print("-" * 50)

[0] ID: 4906
문서 내용: {'total_count': 599, 'id': 4906, 'memb_id': '52a62c11-7c4f-4912-91c9-ff5c145328cd', 'device_desc': '혈당계(N Premier)', 'fullname': '김테스터', 'get_time': '2025-03-01 02:12:42', 'glucosedata': 107, 'when_eat': '식사 후 4시간', 'create_dttm': '2025-03-01 02:12:59'}
메타데이터: {'when_eat': '식사 후 4시간', 'device_desc': '혈당계(N Premier)', 'get_time': '2025-03-01 02:12:42', 'anonymized': False, 'data_type': 'blood_glucose', 'fullname': '김테스터', 'ingest_time': '2025-07-21 14:27:16', 'is_private': True, 'created_by_system': 'FaMiliCare', 'data_source': '혈당계(N Premier)', 'memb_id': '52a62c11-7c4f-4912-91c9-ff5c145328cd', 'consent_granted': True, 'create_dttm': '2025-03-01 02:12:59'}
임베딩 벡터 (길이 768): [ 0.1028674  -0.28704309 -0.05673055 -0.29980794  0.20835578]...
--------------------------------------------------
[1] ID: 4905
문서 내용: {'total_count': 599, 'id': 4905, 'memb_id': '52a62c11-7c4f-4912-91c9-ff5c145328cd', 'device_desc': '혈당계(N Premier)', 'fullname': '김테스터', 'get_time': '2025-02-24 0